# DATASCI 451 - Results Visualization

Key visualizations:
1. Shrinkage Effect
2. Station Effects Forest Plot
3. Month Effects
4. Posterior Predictive Check

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

In [ ]:
# Load traces
trace_cp = az.from_netcdf('data/trace_complete_pooling.nc')
trace_np = az.from_netcdf('data/trace_no_pooling.nc')
trace_hier = az.from_netcdf('data/trace_hierarchical.nc')

# Load data
df = pd.read_csv('data/selected_stations_monthly.csv')
station_mapping = pd.read_csv('data/station_mapping.csv')
station_names = station_mapping['station_name'].values

print(f"Loaded traces and data")
print(f"Stations: {list(station_names)}")

## 1. Shrinkage Effect Plot

The **key visualization** showing how partial pooling shrinks extreme estimates toward the group mean.

In [ ]:
# Extract station effect estimates
alpha_np = trace_np.posterior['alpha'].mean(dim=['chain', 'draw']).values
alpha_hier = trace_hier.posterior['alpha'].mean(dim=['chain', 'draw']).values
mu_alpha = float(trace_hier.posterior['mu_alpha'].mean())

# Observed station means (raw data)
station_means = df.groupby('short_name')['TAVG_mean'].mean().reindex(station_names).values

print("Station Effect Estimates:")
print(f"{'Station':<20} {'Raw Mean':>10} {'No Pool':>10} {'Hier':>10} {'Shrinkage':>10}")
print("-" * 60)
for i, name in enumerate(station_names):
    shrinkage = (alpha_np[i] - alpha_hier[i]) / (alpha_np[i] - mu_alpha) * 100
    print(f"{name:<20} {station_means[i]:>10.1f} {alpha_np[i]:>10.1f} {alpha_hier[i]:>10.1f} {shrinkage:>9.1f}%")

In [ ]:
# Shrinkage Plot
fig, ax = plt.subplots(figsize=(10, 8))

# Diagonal line (no shrinkage)
lims = [min(alpha_np.min(), alpha_hier.min()) - 2, max(alpha_np.max(), alpha_hier.max()) + 2]
ax.plot(lims, lims, 'k--', alpha=0.5, label='No shrinkage')

# Horizontal line at group mean
ax.axhline(mu_alpha, color='red', linestyle=':', alpha=0.7, label=f'Group mean: {mu_alpha:.1f}°F')

# Points with arrows showing shrinkage
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(station_names)))
for i, name in enumerate(station_names):
    ax.scatter(alpha_np[i], alpha_hier[i], s=150, c=[colors[i]], 
               edgecolors='black', linewidths=1.5, zorder=5)
    ax.annotate(name, (alpha_np[i], alpha_hier[i]), 
                xytext=(8, 0), textcoords='offset points', fontsize=9)
    # Arrow showing shrinkage direction
    ax.annotate('', xy=(alpha_np[i], alpha_hier[i]), xytext=(alpha_np[i], alpha_np[i]),
                arrowprops=dict(arrowstyle='->', color='gray', alpha=0.5))

ax.set_xlabel('No Pooling Estimate (°F)', fontsize=12)
ax.set_ylabel('Partial Pooling (Hierarchical) Estimate (°F)', fontsize=12)
ax.set_title('Shrinkage Effect: Station Temperature Estimates\n'
             'Points below diagonal = shrinkage toward group mean', fontsize=14)
ax.legend(loc='upper left')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/12_shrinkage_effect.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation

- Points **on the diagonal**: No shrinkage (estimate unchanged)
- Points **below diagonal** (for values > mean): Shrunk toward group mean
- Points **above diagonal** (for values < mean): Shrunk toward group mean
- **Extreme estimates** (far from group mean) show more shrinkage

## 2. Forest Plot: Station Effects with Credible Intervals

In [ ]:
# Extract credible intervals
def get_ci(trace, var, idx=None, hdi_prob=0.95):
    if idx is not None:
        data = trace.posterior[var].sel(alpha_dim_0=idx)
    else:
        data = trace.posterior[var]
    mean = float(data.mean())
    hdi = az.hdi(data, hdi_prob=hdi_prob)
    return mean, float(hdi.min()), float(hdi.max())

# Get CIs for both models
results = []
for i, name in enumerate(station_names):
    # No pooling
    np_mean, np_lo, np_hi = get_ci(trace_np, 'alpha', i)
    # Hierarchical
    hier_mean, hier_lo, hier_hi = get_ci(trace_hier, 'alpha', i)
    
    results.append({
        'station': name,
        'np_mean': np_mean, 'np_lo': np_lo, 'np_hi': np_hi, 'np_width': np_hi - np_lo,
        'hier_mean': hier_mean, 'hier_lo': hier_lo, 'hier_hi': hier_hi, 'hier_width': hier_hi - hier_lo
    })

ci_df = pd.DataFrame(results)
ci_df['ci_reduction'] = ((ci_df['np_width'] - ci_df['hier_width']) / ci_df['np_width'] * 100).round(1)
print(ci_df[['station', 'np_width', 'hier_width', 'ci_reduction']].to_string(index=False))

In [ ]:
# Forest Plot
fig, ax = plt.subplots(figsize=(12, 8))

y_positions = np.arange(len(station_names))
height = 0.35

# No pooling (blue)
for i, row in ci_df.iterrows():
    ax.errorbar(row['np_mean'], y_positions[i] + height/2, 
                xerr=[[row['np_mean'] - row['np_lo']], [row['np_hi'] - row['np_mean']]],
                fmt='o', color='steelblue', markersize=8, capsize=4, capthick=2,
                label='No Pooling' if i == 0 else '')

# Hierarchical (orange)
for i, row in ci_df.iterrows():
    ax.errorbar(row['hier_mean'], y_positions[i] - height/2,
                xerr=[[row['hier_mean'] - row['hier_lo']], [row['hier_hi'] - row['hier_mean']]],
                fmt='s', color='coral', markersize=8, capsize=4, capthick=2,
                label='Hierarchical' if i == 0 else '')

# Group mean line
ax.axvline(mu_alpha, color='red', linestyle='--', alpha=0.7, label=f'Population mean: {mu_alpha:.1f}°F')

ax.set_yticks(y_positions)
ax.set_yticklabels(station_names)
ax.set_xlabel('Station Effect α (°F)', fontsize=12)
ax.set_title('Station Effects: No Pooling vs Hierarchical\n(95% Credible Intervals)', fontsize=14)
ax.legend(loc='upper right')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/13_forest_plot_station_effects.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation

- **Hierarchical CIs are narrower**: Borrowing strength reduces uncertainty
- **Extreme stations shrink more**: Bergland Dam (coldest) and Ann Arbor (warmest) show larger shrinkage
- **All stations pulled toward population mean**: The red dashed line

## 3. Month Effects

In [ ]:
# Extract month effects from hierarchical model
beta_samples = trace_hier.posterior['beta'].values.reshape(-1, 4)  # (samples, months)

month_names = ['January', 'February', 'March', 'April']
beta_summary = []
for m in range(4):
    mean = beta_samples[:, m].mean()
    lo, hi = np.percentile(beta_samples[:, m], [2.5, 97.5])
    beta_summary.append({'month': month_names[m], 'mean': mean, 'lo': lo, 'hi': hi})

beta_df = pd.DataFrame(beta_summary)
print(beta_df.to_string(index=False))

In [ ]:
# Month effects plot
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(4)
colors = ['#3498db', '#9b59b6', '#2ecc71', '#e74c3c']

for i, row in beta_df.iterrows():
    ax.bar(x[i], row['mean'], color=colors[i], alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.errorbar(x[i], row['mean'], yerr=[[row['mean'] - row['lo']], [row['hi'] - row['mean']]],
                fmt='none', color='black', capsize=6, capthick=2)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(month_names)
ax.set_ylabel('Month Effect β (°F)', fontsize=12)
ax.set_title('Seasonal Temperature Effects\n(Relative to Overall Mean, 95% CI)', fontsize=14)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, row in beta_df.iterrows():
    ax.annotate(f"{row['mean']:+.1f}°F", xy=(x[i], row['mean']),
                xytext=(0, 10 if row['mean'] > 0 else -15), textcoords='offset points',
                ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/14_month_effects.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation

- **Clear seasonal pattern**: Temperature increases from January to April
- **January is coldest**: β₁ ≈ -10°F below average
- **April is warmest**: β₄ ≈ +10°F above average
- **All CIs exclude zero**: Seasonal effects are statistically significant

## 4. Posterior Predictive Check

In [ ]:
# Generate posterior predictions
import pymc as pm

# Recreate data
station_idx = {name: i for i, name in enumerate(station_names)}
df['station_id'] = df['short_name'].map(station_idx)
df['month_id'] = df['Month'] - 1
y = df['TAVG_mean'].values
station = df['station_id'].values
month = df['month_id'].values

# Get posterior means
alpha_post = trace_hier.posterior['alpha'].mean(dim=['chain', 'draw']).values
beta_post = trace_hier.posterior['beta'].mean(dim=['chain', 'draw']).values
sigma_post = float(trace_hier.posterior['sigma'].mean())

# Predicted values
y_pred = alpha_post[station] + beta_post[month]

# Residuals
residuals = y - y_pred

In [ ]:
# Posterior predictive plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Observed vs Predicted
ax1 = axes[0]
ax1.scatter(y_pred, y, alpha=0.7, s=80, edgecolors='black', linewidths=0.5)
lims = [min(y.min(), y_pred.min()) - 2, max(y.max(), y_pred.max()) + 2]
ax1.plot(lims, lims, 'r--', label='Perfect prediction')
ax1.set_xlabel('Predicted Temperature (°F)')
ax1.set_ylabel('Observed Temperature (°F)')
ax1.set_title('Observed vs Predicted')
ax1.legend()
ax1.set_xlim(lims)
ax1.set_ylim(lims)
ax1.set_aspect('equal')

# Plot 2: Residual distribution
ax2 = axes[1]
ax2.hist(residuals, bins=15, edgecolor='black', alpha=0.7, density=True)
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
from scipy.stats import norm
ax2.plot(x_norm, norm.pdf(x_norm, 0, sigma_post), 'r-', linewidth=2, 
         label=f'N(0, {sigma_post:.1f}²)')
ax2.set_xlabel('Residual (°F)')
ax2.set_ylabel('Density')
ax2.set_title('Residual Distribution')
ax2.legend()

# Plot 3: Residuals by station
ax3 = axes[2]
df['residual'] = residuals
df.boxplot(column='residual', by='short_name', ax=ax3)
ax3.axhline(0, color='red', linestyle='--')
ax3.set_xlabel('Station')
ax3.set_ylabel('Residual (°F)')
ax3.set_title('Residuals by Station')
plt.suptitle('')  # Remove automatic title
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('plots/15_posterior_predictive_check.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Model fit statistics
from sklearn.metrics import r2_score, mean_absolute_error

r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(np.mean(residuals**2))

print("Model Fit Statistics (Hierarchical Model):")
print(f"  R²: {r2:.4f}")
print(f"  MAE: {mae:.2f}°F")
print(f"  RMSE: {rmse:.2f}°F")
print(f"  σ (posterior): {sigma_post:.2f}°F")

## 5. Summary Table for Report

In [ ]:
# Create summary table
print("="*70)
print("PARAMETER ESTIMATES SUMMARY (Hierarchical Model)")
print("="*70)

# Hyperparameters
print("\nHyperparameters:")
print(f"  μ_α (population mean): {mu_alpha:.2f}°F")
tau = float(trace_hier.posterior['tau'].mean())
print(f"  τ (between-station SD): {tau:.2f}°F")
print(f"  σ (observation noise): {sigma_post:.2f}°F")

# Station effects
print("\nStation Effects (α_i):")
print(f"  {'Station':<20} {'Mean':>8} {'95% CI':>20}")
for i, name in enumerate(station_names):
    row = ci_df.iloc[i]
    print(f"  {name:<20} {row['hier_mean']:>8.2f} [{row['hier_lo']:.2f}, {row['hier_hi']:.2f}]")

# Month effects
print("\nMonth Effects (β_m):")
for i, row in beta_df.iterrows():
    print(f"  {row['month']:<10} {row['mean']:>+8.2f} [{row['lo']:.2f}, {row['hi']:.2f}]")

In [ ]:
# Key findings
print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)
print(f"""
1. SEASONAL PATTERN
   - Temperature increases monotonically from January to April
   - Range: {beta_df['mean'].min():.1f}°F (Jan) to {beta_df['mean'].max():.1f}°F (Apr)
   - Total seasonal swing: {beta_df['mean'].max() - beta_df['mean'].min():.1f}°F

2. GEOGRAPHIC GRADIENT
   - Clear north-south temperature gradient
   - Coldest: {station_names[ci_df['hier_mean'].argmin()]} ({ci_df['hier_mean'].min():.1f}°F)
   - Warmest: {station_names[ci_df['hier_mean'].argmax()]} ({ci_df['hier_mean'].max():.1f}°F)
   - Range across stations: {ci_df['hier_mean'].max() - ci_df['hier_mean'].min():.1f}°F

3. HIERARCHICAL MODEL BENEFITS
   - Between-station variation (τ): {tau:.2f}°F
   - Within-station noise (σ): {sigma_post:.2f}°F  
   - Ratio τ/σ = {tau/sigma_post:.2f} → Station differences are substantial
   - Average CI reduction: {ci_df['ci_reduction'].mean():.1f}%

4. MODEL FIT
   - R² = {r2:.3f} (explains {r2*100:.1f}% of variance)
   - RMSE = {rmse:.2f}°F
""")

---
## Next Steps

1. Copy key figures to report
2. Include parameter tables
3. Write interpretation sections